# Task 1 — LoRA Fine-Tuning for Machine Translation (WMT16 EN↔TR)

**Önerilen runtime:** A100 40GB (~1.5–2 saat, ~25 CU). Fallback: L4 / V100 da çalışır, training daha uzun sürer.

Pipeline (sırayla):
1. Drive mount + HF cache Drive'a → modeller bir kez indirilir, sonraki session'lar oradan okur
2. Repo clone + pip install
3. Model prefetch (Qwen2.5-7B; ilk session ~10–15 dk Drive'a yazma)
4. WMT16 EN↔TR veri hazırlığı
5. QLoRA training (checkpoint Drive'a, resume desteği)
6. Inference (test set üzerinde batched generation)
7. COMET değerlendirme + HW2 baseline karşılaştırma
8. Sonuçları zip + Drive'dan lokale indir

In [ ]:
# === Drive mount + HF cache Drive'a ===
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/llm_final'
os.environ['DRIVE_ROOT']         = DRIVE_ROOT
os.environ['HF_HOME']            = f'{DRIVE_ROOT}/hf_cache'
os.environ['HF_DATASETS_CACHE']  = f'{DRIVE_ROOT}/hf_cache/datasets'

for sub in ('hf_cache', 'data', 'models/checkpoints', 'results', 'exports'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

print('Drive root:', DRIVE_ROOT)

In [ ]:
# === GPU doğrulama ===
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# === Repo clone + bağımlılıklar (HW2-style: torch'a dokunma) ===
REPO_URL = 'https://github.com/mustafagalata/lora-project.git'  # << kendi repo URL'ini yaz
%cd /content
!rm -rf /content/repo && git clone {REPO_URL} /content/repo
%cd /content/repo

# Temel paketler — torch ve torchvision'a açıkça dokunmuyoruz (Colab uyumlu pair kalır)
!pip install -q \
    transformers \
    peft \
    trl \
    accelerate \
    bitsandbytes \
    datasets \
    sentence-transformers \
    "faiss-cpu>=1.8.0" \
    langchain langchain-community langchain-huggingface langchain-text-splitters \
    pypdf pdfplumber tiktoken \
    pyyaml numpy pandas tqdm matplotlib

# COMET: --no-deps ile torch bağımlılığını bypass et, sonra eksik bağımlılıkları manuel ekle.
# `entmax` COMET'in LayerwiseAttention modülü için zorunlu.
!pip install -q unbabel-comet --no-deps
!pip install -q "lightning>=2.0" torchmetrics sentencepiece entmax

# Task 3 design (sadece tasarım — runtime'da kullanılmıyor ama hafif)
!pip install -q langdetect wikipedia-api

# NOT: flash-attn kurulmuyor — build 5-15 dk sürüyor ve ~%20 hızlanma için
# o süreyi vermek değmez. model_loader.load_base_model() otomatik SDPA'ya düşer.

In [ ]:
# === Model prefetch (ilk session'da Drive'a yazılır; sonraki run'larda no-op) ===
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-7B-Instruct')   # ~15GB, ilk seferinde 10-15dk
# COMET modelini evaluate_comet hücresi kendi indirir

## 1) WMT16 EN↔TR veri hazırlığı

50K subsample → JSONL (`$DRIVE_ROOT/data/wmt16_en_tr/{train,validation,test}.jsonl`).

In [ ]:
!python -m src.task1_lora_mt.prepare_data

## 2) QLoRA training

Batch boyutunu `config.yaml` içinde `task1.training.per_device_batch_size` ile ayarla:
- **A100 40GB:** 4 (default; effective batch = 16 grad_accum=4 ile)
- **L4 24GB:**   2
- **T4 16GB:**   1

Checkpoint Drive'a (`$DRIVE_ROOT/models/checkpoints/qwen25_7b_lora_mt`). Kesinti olursa hücreye `--resume` ekleyerek yeniden çalıştır — en son checkpoint'ten devam eder.

### ⚡ Smoke test (önerilen, opsiyonel)

Full training (1-2 saat) öncesi 5 step'lik bir mini-run; sürüm/uyumluluk hatalarını (trl API, peft, SFTConfig) ~30 saniyede yakalar. Geçerse bir alttaki argümansız hücreyi çalıştır.

In [ ]:
!python -m src.task1_lora_mt.train_lora --max_steps 5 --save_steps 5

In [ ]:
!python -m src.task1_lora_mt.train_lora

## 3) Inference (WMT16 test set, HW2 paritesi)

Adapter `adapter_final/` altından otomatik yüklenir. `config.yaml`'daki `task1.test_limit_per_direction` her yöne uygulanır (default: 500 → toplam 1000 örnek). Hızlı smoke test için config'i geçici düşürebilirsin (`-1` = full test).

In [ ]:
!python -m src.task1_lora_mt.inference

## 4) COMET değerlendirme

`wmt22-comet-da` ile yön bazında (en2tr, tr2en) sistem skoru; HW2 baseline çıktıya eklenir.

In [ ]:
!python -m src.task1_lora_mt.evaluate_comet

## 5) Sonuçları zip + lokale indir

Drive'daki `exports/task1_artifacts.zip` dosyasını web arayüzünden indir.

In [ ]:
import os, shutil
src_results = f'{DRIVE_ROOT}/results'
src_adapter = f'{DRIVE_ROOT}/models/checkpoints/qwen25_7b_lora_mt/adapter_final'
out_dir     = f'{DRIVE_ROOT}/exports/task1_artifacts'
shutil.rmtree(out_dir, ignore_errors=True)
os.makedirs(out_dir, exist_ok=True)
if os.path.exists(src_results):
    shutil.copytree(src_results, f'{out_dir}/results')
if os.path.exists(src_adapter):
    shutil.copytree(src_adapter, f'{out_dir}/adapter_final')
shutil.make_archive(out_dir, 'zip', out_dir)
print('Wrote:', out_dir + '.zip')